# ARMR in PyHealth: medication recommendation on MIMIC-III

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/armr_mimic3_drug_recommendation.ipynb)

This notebook runs **ARMR** (Feiyue Wu, Tianxing Wu and Shenqi Jing, *ARMR: Adaptively Responsive Network for Medication Recommendation*, IJCAI 2025, [arXiv:2507.04428](https://arxiv.org/abs/2507.04428)) as implemented in PyHealth (`pyhealth.models.ARMR`), and compares it with PyHealth's GAMENet, SafeDrug and RETAIN on the same task, split and metrics.

ARMR has three parts:
- **Piecewise temporal learning.** The most recent visits go through a feedforward network and the earlier ones through a Mamba block; the recent states attend over the distant ones.
- **Adaptively responsive module.** The medication history is split into drugs that were reused and drugs that were new at each visit. Each drug's embedding is weighted by whether the patient took it before, with a learned time decay.
- **Joint recommendation.** A direct score from the patient representation is fused with the cosine similarity between the patient and each weighted drug embedding. The loss is 0.7 × BCE + 0.3 × multi-label margin.

**How to use it.** Part 1 runs end to end on PyHealth's open synthetic MIMIC-III, with no credentials. Part 2 repeats the experiment on real MIMIC-III v1.4: set `MIMIC3_ROOT` to your copy (PhysioNet credentials required) and run it.

## Setup

On Colab this installs PyHealth from the branch that adds ARMR. After the pull request is merged, install `pyhealth` from PyPI or from `master` instead.

In [1]:
import sys
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/solarsys/PyHealth@feat/armr"],
        check=True,
    )

In [2]:
import random

import numpy as np
import pandas as pd
import torch

from pyhealth.datasets import MIMIC3Dataset, get_dataloader, split_by_patient
from pyhealth.medcode import ATC
from pyhealth.metrics import ddi_rate_score
from pyhealth.models import ARMR, GAMENet, RETAIN, SafeDrug
from pyhealth.tasks import DrugRecommendationMIMIC3
from pyhealth.trainer import Trainer

METRICS = ["jaccard_samples", "pr_auc_samples", "f1_samples"]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def ddi_matrix(samples) -> np.ndarray:
    """Drug-drug interactions between the task's ATC-3 drugs (as GAMENet and SafeDrug)."""
    vocab = samples.output_processors["drugs"].label_vocab
    adj = np.zeros((len(vocab), len(vocab)))
    for a, b in ATC().get_ddi(gamenet_ddi=True):
        a, b = ATC.convert(a, level=3), ATC.convert(b, level=3)
        if a in vocab and b in vocab:
            adj[vocab[a], vocab[b]] = adj[vocab[b], vocab[a]] = 1
    return adj


def load_samples(root: str, dev: bool = False):
    """Loads MIMIC-III and builds PyHealth's drug-recommendation samples."""
    dataset = MIMIC3Dataset(
        root=root,
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        dev=dev,
    )
    return dataset.set_task(DrugRecommendationMIMIC3())


MODELS = {
    # ARMR's defaults follow the paper; lr 2e-4 is the authors' setting
    "ARMR": (lambda s: ARMR(s, embedding_dim=64, n_recent=3), 2e-4),
    "GAMENet": (lambda s: GAMENet(s), 1e-3),
    "SafeDrug": (lambda s: SafeDrug(s), 1e-3),
    "RETAIN": (lambda s: RETAIN(s), 1e-3),
}


def compare(samples, epochs: int, seed: int = 42, batch_size: int = 64):
    """Trains each model on one patient-level split and scores it on the test patients."""
    train, val, test = split_by_patient(samples, [0.7, 0.1, 0.2], seed=seed)
    loaders = (
        get_dataloader(train, batch_size=batch_size, shuffle=True),
        get_dataloader(val, batch_size=batch_size, shuffle=False),
        get_dataloader(test, batch_size=batch_size, shuffle=False),
    )
    ddi = ddi_matrix(samples)
    rows = {}
    for name, (make, lr) in MODELS.items():
        seed_everything(seed)
        trainer = Trainer(model=make(samples), metrics=METRICS, enable_logging=False)
        trainer.train(
            train_dataloader=loaders[0], val_dataloader=loaders[1], epochs=epochs,
            optimizer_params={"lr": lr}, monitor="jaccard_samples",
        )
        scores = trainer.evaluate(loaders[2])
        # DDI rate of the recommended sets (probability >= 0.5)
        _, y_prob, _ = trainer.inference(loaders[2])
        recommended = [np.flatnonzero(row >= 0.5) for row in y_prob]
        rows[name] = {
            "Jaccard": scores["jaccard_samples"],
            "PRAUC": scores["pr_auc_samples"],
            "F1": scores["f1_samples"],
            "DDI rate": ddi_rate_score(recommended, ddi),
        }
    return (pd.DataFrame(rows).T * 100).round(2)

## Part 1: open synthetic MIMIC-III (no credentials)

PyHealth hosts a synthetic MIMIC-III with the real schema. It is useful for checking that everything runs; its scores say nothing about the paper's results. Three epochs take a few minutes on a CPU. That is too short for ARMR, which trains at the authors' learning rate (2e-4, five times lower than the baselines'), so it trails here; the comparison that matters is Part 2's.

In [3]:
synthetic = load_samples("https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III")
print(f"{len(synthetic)} samples")
synthetic_results = compare(synthetic, epochs=3)
synthetic_results

1858 samples


,Jaccard,PRAUC,F1,DDI rate
ARMR,24.34,46.17,36.24,6.46
GAMENet,30.77,56.33,43.83,4.51
SafeDrug,31.19,55.40,44.27,8.20
RETAIN,28.75,53.94,41.28,7.46


## Part 2: MIMIC-III v1.4 (credentialed)

Set `MIMIC3_ROOT` to the folder with `ADMISSIONS.csv.gz`, `DIAGNOSES_ICD.csv.gz`, `PROCEDURES_ICD.csv.gz` and `PRESCRIPTIONS.csv.gz` (for example a mounted Google Drive folder). On a Colab GPU, 50 epochs take roughly an hour for all four models.

The paper reports, on its MIMIC-III and MIMIC-IV cohorts (Table 2, %):

| Dataset | Jaccard | PRAUC | F1 |
|---|---|---|---|
| MIMIC-III | 54.51 | 79.63 | 69.59 |
| MIMIC-IV | 51.74 | 76.95 | 67.12 |

The paper builds its cohort with the Carmen preprocessing pipeline (ATC level 3, 131 drugs, patients filtered by visit count), while PyHealth's `DrugRecommendationMIMIC3` uses its own filtering and drug vocabulary. Expect the same ordering of models, not identical numbers.

In [4]:
MIMIC3_ROOT = ""  # e.g. "/content/drive/MyDrive/mimic-iii-clinical-database-1.4"

if MIMIC3_ROOT:
    mimic3 = load_samples(MIMIC3_ROOT)
    print(f"{len(mimic3)} samples")
    mimic3_results = compare(mimic3, epochs=50)
    display(mimic3_results)
else:
    print("Set MIMIC3_ROOT to run the experiment on MIMIC-III.")

Set MIMIC3_ROOT to run the experiment on MIMIC-III.


## Notes for reproducing the paper

- `ARMR(..., use_graph=True)` (the default) adds GCN embeddings of the drug co-occurrence and drug-drug interaction graphs to the drug embeddings, as the authors' released code does. `use_graph=False` gives the equations exactly as written in the paper.
- Three details the paper leaves open, and the choices made: Eq. 9's time decay is one learned scale and rate; Eq. 11's attention gives a weight per embedding feature; the "recent" segment is the N visits closest to the prediction.
- Questions or corrections: please open an issue at https://github.com/sunlabuiuc/PyHealth or write on the PyHealth Discord.